# GroupDRO eta=0.1 + 2 epochs

Less aggressive GroupDRO with more training time.

In [1]:
import os, json, numpy as np, pandas as pd, torch, joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

np.random.seed(42); torch.manual_seed(42)

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
df_train = pd.read_csv(BASE_DIR / "data/processed/train.csv")
df_val = pd.read_csv(BASE_DIR / "data/processed/val.csv")
df_test = pd.read_csv(BASE_DIR / "data/processed/test.csv")

mapping = pd.read_csv(BASE_DIR / "data/processed/label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]: df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)
print(f"Train: {len(df_train)}, Labels: {num_labels}")

Train: 16530, Labels: 9


In [3]:
# Sqrt reweighting
city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)

# City IDs
city_to_id = {c: i for i, c in enumerate(df_train["city_group"].unique())}
num_groups = len(city_to_id)
df_train["city_id"] = df_train["city_group"].map(city_to_id).astype(int)
df_val["city_id"] = df_val["city_group"].map(city_to_id).fillna(-1).astype(int)
df_test["city_id"] = df_test["city_group"].map(city_to_id).fillna(-1).astype(int)
print(f"Groups: {num_groups}")

Groups: 41


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize(b): return tokenizer(b["resume_text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_pandas(df_train[["resume_text", "y", "city_id", "sample_weight"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text", "y", "city_id"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text", "y", "city_id"]]).map(tokenize, batched=True)

for ds in [train_ds, val_ds, test_ds]: ds.rename_column("y", "labels")
train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id", "sample_weight"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])

Map: 100%|██████████| 5510/5510 [01:09<00:00, 78.87 examples/s] 


In [5]:
class GroupDROTrainer(Trainer):
    def __init__(self, *args, num_groups, eta=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_groups, self.eta = num_groups, eta
        self.q = torch.ones(num_groups) / num_groups
        print(f"GroupDRO: {num_groups} groups, eta={eta}")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        city_id = inputs.pop("city_id")
        sample_weight = inputs.pop("sample_weight", None)
        outputs = model(**inputs)
        
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        per_sample_loss = loss_fct(outputs.logits, inputs["labels"])
        if sample_weight is not None:
            per_sample_loss = per_sample_loss * sample_weight.to(per_sample_loss.dtype)

        device = per_sample_loss.device
        group_loss = torch.zeros(self.num_groups, device=device)
        group_count = torch.zeros(self.num_groups, device=device)
        for g in range(self.num_groups):
            mask = (city_id == g)
            if mask.any():
                group_loss[g] = per_sample_loss[mask].mean()
                group_count[g] = mask.sum().float()

        with torch.no_grad():
            q = self.q.to(device) * torch.exp(self.eta * group_loss * (group_count > 0).float())
            self.q = (q / q.sum()).cpu()

        loss = (self.q.to(device) * group_loss).sum()
        return (loss, outputs) if return_outputs else loss

In [6]:
MODEL_NAME = "bert_gdro_eta01_2ep"
ETA = 0.1

args = TrainingArguments(
    output_dir=f"./models/{MODEL_NAME}",
    learning_rate=2e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=2, weight_decay=0.01, logging_steps=100,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy", remove_unused_columns=False
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "macro_f1": f1_score(p.label_ids, preds, average="macro")}

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
trainer = GroupDROTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          compute_metrics=compute_metrics, num_groups=num_groups, eta=ETA)

print(f"Training: GroupDRO eta={ETA}, 2 epochs")
trainer.train()

Loading weights: 100%|██████████| 199/199 [00:03<00:00, 54.90it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those par

GroupDRO: 41 groups, eta=0.1
Training: GroupDRO eta=0.1, 2 epochs


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.234834,1.215253,0.501996,0.509029
2,0.213308,1.139691,0.553358,0.560197


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.13s/it]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.10s/it]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.La

TrainOutput(global_step=4134, training_loss=0.24231690995982724, metrics={'train_runtime': 11694.4703, 'train_samples_per_second': 2.827, 'train_steps_per_second': 0.354, 'total_flos': 2174749547351040.0, 'train_loss': 0.24231690995982724, 'epoch': 2.0})

In [7]:
preds = trainer.predict(test_ds)
y_true, y_pred = preds.label_ids, np.argmax(preds.predictions, axis=1)

print(f"\nTEST RESULTS")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"Macro F1: {f1_score(y_true, y_pred, average='macro'):.4f}")

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



TEST RESULTS
Accuracy: 0.5477
Macro F1: 0.5525


In [8]:
# Fairness
df_eval = df_test.copy()
df_eval["y_true"], df_eval["y_pred"] = y_true, y_pred
df_eval["correct"] = df_eval["y_true"] == df_eval["y_pred"]

def ovr_rates(df, grp, nc):
    groups = sorted(df[grp].dropna().unique())
    tpr, support = np.zeros((len(groups), nc)), np.zeros((len(groups), nc))
    for gi, g in enumerate(groups):
        dg = df[df[grp] == g]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(nc):
            pm = (yt == c)
            TP, FN = np.sum((yp == c) & pm), np.sum((yp != c) & pm)
            support[gi, c] = pm.sum()
            tpr[gi, c] = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    return tpr, support

def gaps(tpr, sup, ms=30):
    g = []
    for c in range(tpr.shape[1]):
        col = tpr[sup[:, c] >= ms, c] if ms else tpr[:, c]
        col = col[~np.isnan(col)]
        g.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    g = np.array(g)
    v = g[~np.isnan(g)]
    return v.max() if len(v) else np.nan, v.mean() if len(v) else np.nan

tpr, sup = ovr_rates(df_eval, "city_group", num_labels)
tw, tm = gaps(tpr, sup, 30)
print(f"\nFAIRNESS (robust n>=30): worst={tw:.4f}, macro={tm:.4f}")
print(f"Compare: best=0.609 acc, 0.329 worst, 0.116 macro")


FAIRNESS (robust n>=30): worst=0.2647, macro=0.1130
Compare: best=0.609 acc, 0.329 worst, 0.116 macro


In [9]:
SAVE_DIR = Path("models") / MODEL_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
joblib.dump(le, SAVE_DIR / "label_encoder.joblib")

cfg = {"method": f"GroupDRO eta={ETA} + sqrt_rw, 2ep", "eta": ETA,
       "accuracy": float(accuracy_score(y_true, y_pred)),
       "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
       "tpr_gap_worst_robust": float(tw), "tpr_gap_macro_robust": float(tm)}
with open(SAVE_DIR / "training_config.json", "w") as f: json.dump(cfg, f, indent=2)
print(f"Saved: {SAVE_DIR}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.27it/s]

Saved: models/bert_gdro_eta01_2ep
